In [ ]:
import time, math, csv
import numpy as np
import board, busio, sys, signal, select

import adafruit_ads1x15.ads1115 as ADS
from adafruit_ads1x15.analog_in import AnalogIn

import threading
import cv2
from picamera2 import Picamera2
from ultralytics import YOLO

# ========================= Mouse logging via evdev ==================
# sudo apt install -y python3-evdev
from evdev import InputDevice, ecodes
import os
# ===================================================================

# ========================= Graph generation =========================
import matplotlib
matplotlib.use("Agg")  # headless-safe (no GUI needed on Pi)
import matplotlib.pyplot as plt
# ===================================================================


# =========================================================
#                 ADC + MICS (2 ADS)
# =========================================================
i2c = busio.I2C(board.SCL, board.SDA)

ads1 = ADS.ADS1115(i2c, address=0x48)
ads2 = ADS.ADS1115(i2c, address=0x49)

ads1.gain = 1
ads2.gain = 1
ads1.data_rate = 860
ads2.data_rate = 860

try:
    ads1.mode = ADS.Mode.CONTINUOUS
    ads2.mode = ADS.Mode.CONTINUOUS
except Exception:
    pass

mic1 = AnalogIn(ads1, 0)   # ADS1 A0
mic2 = AnalogIn(ads2, 0)   # ADS2 A0

MIC_SAMPLES_PER_TICK = 120


def measure_db_fast(channel, n_samples=MIC_SAMPLES_PER_TICK):
    s = 0.0
    ss = 0.0
    for _ in range(n_samples):
        x = float(channel.value)
        s += x
        ss += x * x

    mean = s / n_samples
    var = (ss / n_samples) - (mean * mean)
    if var < 1e-9:
        return -120.0, 0.0

    rms = math.sqrt(var)
    dbfs = 20.0 * math.log10(rms / 16384.0)
    return dbfs, rms


# =========================================================
#                 Camera / YOLO settings
# =========================================================
MODEL_NAME = "yolov8n.pt"
CONF_THRESH = 0.4
IOU_THRESH = 0.45
PERSON_CLASS_ID = 0

SHOW_WINDOW = True
CAM_SIZE = (320, 240)
EXPOSURE_US = 6000
ANALOG_GAIN = 8.0

YOLO_FPS = 20.0
YOLO_PERIOD = 1.0 / YOLO_FPS


# =========================================================
#            OUTPUT FILES
# =========================================================
CSV_FILENAME = "Final_Stomping_30.csv"
PLOT_FILENAME = "Final_graph_30.png"
OVERLAY_PLOT_FILENAME = "Overlay_graph_30png"

# NEW: camera history plot filename
CAM_HISTORY_PLOT_FILENAME = "Camera_history.png"


# =========================================================
#            Buffers
# =========================================================
rms1_values, db1_values = [], []
rms2_values, db2_values = [], []

time_sec_values = []
human_values = []
mouse_values = []

# NEW: high-rate camera history buffers (~YOLO_FPS)
cam_hist_lock = threading.Lock()
cam_hist_time = []      # seconds since START_TIME
cam_hist_detect = []    # 0/1 per YOLO inference
START_TIME = None


# =========================================================
#            Shared flags / locks
# =========================================================
stop_flag = False

cam_any_lock = threading.Lock()
cam_any_detected = False

mouse_lock = threading.Lock()
mouse_clicked = False

MOUSE_EVENT_PATH = "/dev/input/by-id/usb-YICHIP_Wireless_Device-if01-event-mouse"


def finalize_csv():
    if len(rms1_values) == 0:
        return

    sd_rms1 = float(np.std(rms1_values, ddof=0))
    sd_db1  = float(np.std(db1_values,  ddof=0))
    sd_rms2 = float(np.std(rms2_values, ddof=0))
    sd_db2  = float(np.std(db2_values,  ddof=0))

    with open(CSV_FILENAME, mode="a", newline="") as f:
        w = csv.writer(f)
        w.writerow([])
        w.writerow(["==== FINAL STANDARD DEVIATIONS ===="])
        w.writerow(["Mic1 RMS SD", sd_rms1])
        w.writerow(["Mic1 dBFS SD", sd_db1])
        w.writerow(["Mic2 RMS SD", sd_rms2])
        w.writerow(["Mic2 dBFS SD", sd_db2])

    print("\n========= FINAL STATISTICS =========")
    print(f"Total samples: {len(rms1_values)}")
    print(f"Mic1 RMS SD:   {sd_rms1:.4f}")
    print(f"Mic1 dBFS SD:  {sd_db1:.4f}")
    print(f"Mic2 RMS SD:   {sd_rms2:.4f}")
    print(f"Mic2 dBFS SD:  {sd_db2:.4f}")
    print("===================================\n")


def save_subplot_figure():
    """4-subplot PNG."""
    if len(time_sec_values) == 0:
        return

    t  = np.array(time_sec_values, dtype=float)
    y1 = np.array(rms1_values, dtype=float)
    y2 = np.array(rms2_values, dtype=float)
    yh = np.array(human_values, dtype=float)
    ym = np.array(mouse_values, dtype=float)

    fig, axs = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

    axs[0].plot(t, y1, color="tab:blue", linewidth=1.5)
    axs[0].set_title("Mic 1 RMS")
    axs[0].set_ylabel("RMS")
    axs[0].grid(True, alpha=0.25)

    axs[1].plot(t, y2, color="tab:orange", linewidth=1.5)
    axs[1].set_title("Mic 2 RMS")
    axs[1].set_ylabel("RMS")
    axs[1].grid(True, alpha=0.25)

    axs[2].step(t, yh, where="post", color="tab:green", linewidth=2.0)
    axs[2].set_title("YOLO Human Detection (1 Hz aggregated)")
    axs[2].set_ylabel("Human (0/1)")
    axs[2].set_ylim(-0.1, 1.1)
    axs[2].grid(True, alpha=0.25)

    axs[3].step(t, ym, where="post", color="tab:red", linewidth=2.0)
    axs[3].set_title("Mouse Click (1 Hz)")
    axs[3].set_ylabel("Click (0/1)")
    axs[3].set_ylim(-0.1, 1.1)
    axs[3].set_xlabel("Time (seconds)")
    axs[3].grid(True, alpha=0.25)

    plt.tight_layout()
    fig.savefig(PLOT_FILENAME, dpi=300)
    plt.close(fig)

    print(f"[PLOT] Saved subplot figure to: {PLOT_FILENAME}")


def save_overlay_figure():
    """
    One PNG that overlays:
      - Mic1 RMS
      - Mic2 RMS
      - YOLO detection (1 Hz aggregated)
      - Mouse click (1 Hz)
    RMS on left y-axis, Human/Mouse (0/1) on right y-axis.
    """
    if len(time_sec_values) == 0:
        return

    t  = np.array(time_sec_values, dtype=float)
    y1 = np.array(rms1_values, dtype=float)
    y2 = np.array(rms2_values, dtype=float)
    yh = np.array(human_values, dtype=float)
    ym = np.array(mouse_values, dtype=float)

    fig, ax1 = plt.subplots(figsize=(12, 5))

    l1, = ax1.plot(t, y1, label="Mic1 RMS", color="tab:blue", linewidth=1.8)
    l2, = ax1.plot(t, y2, label="Mic2 RMS", color="tab:orange", linewidth=1.8)
    ax1.set_xlabel("Time (seconds)")
    ax1.set_ylabel("RMS")
    ax1.grid(True, alpha=0.25)

    ax2 = ax1.twinx()
    l3 = ax2.step(t, yh, where="post", label="YOLO Human (1 Hz)", color="tab:green", linewidth=2.2)[0]
    l4 = ax2.step(t, ym, where="post", label="Mouse Click (1 Hz)", color="tab:red", linewidth=2.2)[0]
    ax2.set_ylabel("Human / Mouse (0/1)")
    ax2.set_ylim(-0.1, 1.1)

    ax1.legend([l1, l2, l3, l4],
               [l1.get_label(), l2.get_label(), l3.get_label(), l4.get_label()],
               loc="upper right")

    ax1.set_title("Overlay: Mic1 RMS, Mic2 RMS, YOLO Human, Mouse Click")

    plt.tight_layout()
    fig.savefig(OVERLAY_PLOT_FILENAME, dpi=300)
    plt.close(fig)

    print(f"[PLOT] Saved overlay figure to: {OVERLAY_PLOT_FILENAME}")


def save_camera_history_plot():
    """Separate plot: raw YOLO detection (0/1) vs time at ~YOLO_FPS."""
    with cam_hist_lock:
        if len(cam_hist_time) == 0:
            print("[CAM-HIST] No camera history to plot.")
            return
        t_cam = np.array(cam_hist_time, dtype=float)
        y_cam = np.array(cam_hist_detect, dtype=float)

    fig = plt.figure(figsize=(12, 4))
    plt.step(t_cam, y_cam, where="post", color="tab:purple", linewidth=1.5)
    plt.title(f"Camera Detection History (~{YOLO_FPS:.0f} FPS)")
    plt.ylabel("Detected (0/1)")
    plt.ylim(-0.1, 1.1)
    plt.xlabel("Time (seconds)")
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    fig.savefig(CAM_HISTORY_PLOT_FILENAME, dpi=300)
    plt.close(fig)

    print(f"[CAM-HIST] Saved camera history plot to: {CAM_HISTORY_PLOT_FILENAME}")


def handle_exit(signum, frame):
    global stop_flag
    stop_flag = True
    sys.exit(0)

signal.signal(signal.SIGINT, handle_exit)


class VideoCaptureThread(threading.Thread):
    def __init__(self, picam2):
        super().__init__()
        self.picam2 = picam2
        self.frame = None
        self.running = True

    def run(self):
        while self.running:
            self.frame = self.picam2.capture_array()

    def stop(self):
        self.running = False


def camera_worker():
    global stop_flag, cam_any_detected, START_TIME

    model = YOLO(MODEL_NAME)

    picam2 = Picamera2()
    picam2.preview_configuration.main.size = CAM_SIZE
    picam2.preview_configuration.main.format = "RGB888"
    picam2.configure("preview")
    picam2.start()
    time.sleep(1.0)

    try:
        picam2.set_controls({
            "ExposureTime": int(EXPOSURE_US),
            "AnalogueGain": float(ANALOG_GAIN),
        })
    except Exception as e:
        print(f"[WARN] Could not set camera controls. Details: {e}")

    capture_thread = VideoCaptureThread(picam2)
    capture_thread.start()

    next_yolo = time.time()

    try:
        while not stop_flag:
            if capture_thread.frame is None:
                continue

            now = time.time()
            if now < next_yolo:
                time.sleep(max(0.0, next_yolo - now))
            next_yolo += YOLO_PERIOD

            frame_rgb = capture_thread.frame
            frame_bgr = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)

            results = model.predict(
                source=frame_bgr,
                conf=CONF_THRESH,
                iou=IOU_THRESH,
                classes=[PERSON_CLASS_ID],
                verbose=False
            )

            boxes = results[0].boxes
            raw_detected = boxes is not None and len(boxes) > 0

            # NEW: record high-rate history (~YOLO_FPS)
            if START_TIME is not None:
                with cam_hist_lock:
                    cam_hist_time.append(time.time() - START_TIME)
                    cam_hist_detect.append(1 if raw_detected else 0)

            if raw_detected:
                with cam_any_lock:
                    cam_any_detected = True

            if SHOW_WINDOW:
                if raw_detected:
                    for b in boxes:
                        x1, y1, x2, y2 = map(int, b.xyxy[0].tolist())
                        conf = float(b.conf[0])
                        cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), (0, 255, 0), 2)
                        cv2.putText(
                            frame_bgr, f"Human ({conf:.2f})",
                            (x1, max(20, y1 - 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2
                        )

                cv2.putText(
                    frame_bgr, f"Human detected: {1 if raw_detected else 0}",
                    (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2
                )

                cv2.imshow("YOLO Human Detection", frame_bgr)
                key = cv2.waitKey(1) & 0xFF
                if key == ord("q"):
                    stop_flag = True
                    break

    finally:
        capture_thread.stop()
        capture_thread.join()
        if SHOW_WINDOW:
            cv2.destroyAllWindows()
        picam2.close()


def mouse_worker_evdev():
    global mouse_clicked, stop_flag

    try:
        if not os.path.exists(MOUSE_EVENT_PATH):
            print(f"[MOUSE] ERROR: {MOUSE_EVENT_PATH} not found.")
            print("[MOUSE] Run: ls -l /dev/input/by-id/ | grep -i mouse")
            return

        dev = InputDevice(MOUSE_EVENT_PATH)
        print(f"[MOUSE] Listening on {dev.path} | name='{dev.name}'")

        for event in dev.read_loop():
            if stop_flag:
                break

            if event.type == ecodes.EV_KEY and event.value == 1:
                if event.code in (ecodes.BTN_LEFT, ecodes.BTN_RIGHT, ecodes.BTN_MIDDLE):
                    with mouse_lock:
                        mouse_clicked = True

    except PermissionError:
        print("[MOUSE] Permission denied reading input device.")
        print("[MOUSE] Try running: sudo python3 your_script.py")
        print("[MOUSE] Or add user to input group: sudo usermod -aG input $USER (then reboot)")
    except Exception as e:
        print(f"[MOUSE] Unexpected error: {e}")


def main():
    global stop_flag, cam_any_detected, mouse_clicked, START_TIME

    # Shared time origin for 1 Hz logger + camera history
    START_TIME = time.time()

    cam_thread = threading.Thread(target=camera_worker, daemon=True)
    cam_thread.start()

    mouse_thread = threading.Thread(target=mouse_worker_evdev, daemon=True)
    mouse_thread.start()

    with open(CSV_FILENAME, mode="w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["timestamp", "RMS_mic1", "dBFS_mic1", "RMS_mic2", "dBFS_mic2", "human", "mouse"])

        print("Logging started... Press 'q' (in window) or type q + Enter, or Ctrl+C.\n")

        next_tick = time.time()
        t0 = None

        try:
            while not stop_flag:
                now = time.time()
                if now < next_tick:
                    time.sleep(next_tick - now)
                tick_start = time.time()
                next_tick += 1.0

                db1, rms1 = measure_db_fast(mic1)
                db2, rms2 = measure_db_fast(mic2)

                rms1_values.append(rms1); db1_values.append(db1)
                rms2_values.append(rms2); db2_values.append(db2)

                with cam_any_lock:
                    human_this_second = 1 if cam_any_detected else 0
                    cam_any_detected = False

                with mouse_lock:
                    mouse_this_second = 1 if mouse_clicked else 0
                    mouse_clicked = False

                if t0 is None:
                    t0 = tick_start
                time_sec_values.append(int(round(tick_start - t0)))
                human_values.append(human_this_second)
                mouse_values.append(mouse_this_second)

                timestamp = time.strftime("%Y-%m-%d %H:%M:%S")

                writer.writerow([
                    timestamp,
                    f"{rms1:.4f}", f"{db1:.2f}",
                    f"{rms2:.4f}", f"{db2:.2f}",
                    human_this_second,
                    mouse_this_second
                ])
                f.flush()

                print(
                    f"{timestamp} | "
                    f"Mic1 RMS: {rms1:8.2f} dBFS: {db1:7.2f} | "
                    f"Mic2 RMS: {rms2:8.2f} dBFS: {db2:7.2f} | "
                    f"Human: {human_this_second} | Mouse: {mouse_this_second}"
                )

                if sys.stdin in select.select([sys.stdin], [], [], 0)[0]:
                    line = sys.stdin.readline().strip()
                    if line.lower() == "q":
                        break

                elapsed = time.time() - tick_start
                if elapsed > 1.0:
                    next_tick = time.time() + 1.0

        finally:
            finalize_csv()
            save_subplot_figure()
            save_overlay_figure()
            save_camera_history_plot()  # NEW separate camera history plot
            stop_flag = True


if __name__ == "__main__":
    main()
